![image.png](attachment:image.png)

# imports

In [ ]:
import gymnasium as gym # envrionment
import torch # neural network/loss function
import torch.nn as nn
import torch.optim as optim # optimizer (like GD or Adam)
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
from gymnasium.utils.save_video import save_video
from gymnasium.wrappers import RecordVideo

In Deep RL, there's a need to store experience so the agent won't forget how to handle past situations.

# memory

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity) # double ended quee as its memory (O(1))

    # S, A, R, S_next, done if done
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    # reformats a set number of the agent's memorys
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*batch) # transpose list of tuples
        return np.array(state), action, reward, np.array(next_state), done

    def __len__(self):
        return len(self.buffer)

# deep neural network

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, action_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

## Observation Space (Neural Network's Input)
8-dim vector representing state of the agent
* `[0]` x coordinate of the lander
* `[1]` y coordinate of the lander
* `[2]` linear velocity in x
* `[3]` linear velocity in y
* `[4]` angle of the lander
* `[5]` angular velocity
* `[6]` boolean: left leg touching the ground
* `[7]` boolean: right leg touching the ground

## Action Space (Neural Network's Output)
* `0`: Do nothing
* `1`: Fire left orientation engine
* `2`: Fire main engine
* `3`: Fire right orientation engine

## Reward Structure (Loss)
The mathematical goal is to score at least **+200 points** to officially "solve" the environment
* **Movement:** Positive reward for moving closer to the pad and moving slower; negative reward for tilting.
* **Fuel Efficiency:** Firing the main engine costs -0.3 points per frame; side engines cost -0.03 points. The agent must learn to not waste fuel.
* **Touchdown:** +10 points for each leg making contact with the ground.
* **End Game:** +100 points for a safe landing; -100 points for a crash.

### Discount factor γ:
* Closer to 1: emphasize long term
* Closer to 0: emphasize short term

# world setup/hyperparameter config

In [ ]:
import subprocess

try:
    base_env = gym.make("LunarLander-v3", render_mode="rgb_array")
except Exception as e:
    if "Box2D is not installed" in str(e):
        print("Box2D dependency missing. Installing...")
        subprocess.run(["pip", "install", "swig"])
        subprocess.run(["pip", "install", "\"gymnasium[box2d]\""])
        print("Box2D installed. Please re-run the cell.")
        raise # Re-raise the original error after attempting install to stop execution
    else:
        raise # Re-raise other unexpected errors

every = 10

env = RecordVideo(
    base_env,
    video_folder="training_videos_regular",
    episode_trigger=lambda ep: ep % every == 0
)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy_net = DQN(state_dim, action_dim) # decides actions, updates weights every epoch
target_net = DQN(state_dim, action_dim) # frozen weights, our "ground truth"
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
memory = ReplayBuffer(10000) # max memories of 10000

# smaller on simpler problems (like a grid world)
# bigger on complex problems (like 3D robotics)
batch_size = 64 # network will EXPLODE if we try all at once
gamma = 0.99 # discount factor
epsilon = 1.0 # exploration rate

# smaller decay multiplier would lead it to exploit faster (hovering a lot more to play safe)
# bigger decay multiplier would lead it to be erratic even after hundreds of episodes
epsilon_decay = 0.995 # small decay so it eventually knows to exploit (only after a lot of epochs)
epsilon_min = 0.01 # always have a tiny chance of exploration

Box2D dependency missing. Installing...
Box2D installed. Please re-run the cell.


DependencyNotInstalled: Box2D is not installed, you can install it by run `pip install swig` followed by `pip install "gymnasium[box2d]"`

# training

![image.png](attachment:image.png)

In [ ]:
episodes = 150 # number of iterations
episode_rewards = [] # 1. Create a list to store the rewards

# loop
for episode in range(episodes):
    state, info = env.reset() #
    done = False
    current_ep_reward = 0 # 2. Track the reward for this specific episode

    # epsilon-greedy strategy
    # if random number (0, 1) is less than epsilon, choose random, else, use the NN
    while not done:
        if random.random() < epsilon:
            action = env.action_space.sample() # explore (helps map out physics)
        else:
            # add a fake batch [] -> [[]]
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            # no grad tells PyTorch to turn off its gradient tracking memory
            with torch.no_grad(): # make network give us a prediction (no training)
                action = policy_net(state_tensor).argmax().item() # index of action w/ highest q-value

        # agent takes an action, and it progresses into the next state
        # terminated: crashed/landed
        # truncated: ran out of time (20s)
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # frame reward gets added to the current episode total reward
        current_ep_reward += reward
        # push the entire experience into the ReplayBuffer
        memory.push(state, action, reward, next_state, done)
        # update current state
        state = next_state

        # TIME FOR MATH
        # if we have at least n amount of memories, sample them, and convert to tensors
        # if we had a BS of 1, hard to understnad based on just one past state
        # 128: slower but would be good if we had more CPU power
        if len(memory) > batch_size:
            states, actions, rewards, next_states, dones = memory.sample(batch_size)

            # we unsqueeze bc we pytorch works with matrices, not arrays (adds a fake second dimension)
            states_t = torch.FloatTensor(states)
            actions_t = torch.LongTensor(actions).unsqueeze(1)
            rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
            next_states_t = torch.FloatTensor(next_states)
            dones_t = torch.FloatTensor(dones).unsqueeze(1)

            # when the current state is passes into the policy network,
            # it outptus a matrix of [64, 4], meaning for all 64 states, it
            # gives predicts for all 4 possible actions
            all_q_values = policy_net(states_t)

            # but we only care about the q-value for the action the agent actually took
            # in that past memory
            current_q = all_q_values.gather(1, actions_t)

            # Bellman targets
            #     Q_target = R + gamma*max[Q(S', A')]
            # no torch grad to speed it up
            with torch.no_grad():
                # look at the next 64 states, and give highest q-values
                max_next_q = target_net(next_states_t).max(1)[0].unsqueeze(1)
                # immediate reward and add it to the discounted future maximum
                target_q = rewards_t + gamma * max_next_q * (1 - dones_t)

            # MSE
            loss = nn.MSELoss()(current_q, target_q)

            optimizer.zero_grad() # clear past gradients
            loss.backward() # backpropagation (chain rule)
            optimizer.step() # take a step now yall

    epsilon = max(epsilon_min, epsilon * epsilon_decay) # decay explorate rate
    episode_rewards.append(current_ep_reward) # append reward (will be used for diagnostics)

    if episode % 10 == 0:
        # how the target network learns (between episodes it dont know)
        target_net.load_state_dict(policy_net.state_dict())
        print(f"Episode {episode} | Reward: {current_ep_reward:.2f} | Epsilon: {epsilon:.2f}")


In [ ]:
# # Plotting the results over time
# plt.plot(episode_rewards)
# plt.title("LunarLander Training Progress")
# plt.xlabel("Episode")
# plt.ylabel("Total Reward")
# plt.show()

In [ ]:
# close the training environment to free up memory
env.close()

print("------ LIVE EVALUATION -----:")

# 2. Initialize a new environment specifically for human rendering
eval_env = gym.make("LunarLander-v3", render_mode="human")

eval_episodes = 2

for ep in range(eval_episodes):
    state, info = eval_env.reset()
    done = False
    ep_reward = 0

    while not done:
        # Convert the state to a tensor
        state_tensor = torch.FloatTensor(state).unsqueeze(0)

        # pure exploitation (no randomness)
        with torch.no_grad():
            action = policy_net(state_tensor).argmax().item()

        # take the step
        next_state, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated

        ep_reward += reward
        state = next_state

    print(f"Evaluation Episode {ep + 1} | Final Reward: {ep_reward:.2f}")

eval_env.close()